<a href="https://colab.research.google.com/github/sr606/LLM/blob/main/Mermaid6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#graph_model_replace


class Graph:
    def __init__(self):
        self.stages = {}          # stage_id -> stage object
        self.datasets = set()     # dataset names
        self.stage_edges = []     # (dataset → stage) and (stage → dataset)
        self.column_edges = []    # (source_column, target_column, expression)

    def add_stage(self, stage_id):
        self.stages[stage_id] = {}

    def add_dataset(self, dataset_name):
        self.datasets.add(dataset_name)

    def add_stage_edge(self, source, target):
        self.stage_edges.append((source, target))

    def add_column_edge(self, src_col, tgt_col, expression):
        self.column_edges.append((src_col, tgt_col, expression))



#only build_graph() function
    def build_graph(lineage_response):
    graph = Graph()

    for stage in lineage_response.stages:
        graph.add_stage(stage.stage_id)

        # Inputs → Stage
        for inp in stage.inputs:
            dataset = inp["dataset_name"]
            graph.add_dataset(dataset)
            graph.add_stage_edge(dataset, stage.stage_id)

        # Stage → Outputs
        for out in stage.outputs:
            dataset = out["dataset_name"]
            graph.add_dataset(dataset)
            graph.add_stage_edge(stage.stage_id, dataset)

        # Column lineage
        for col in stage.column_lineage:
            for src in col.source_columns:
                graph.add_column_edge(
                    src,
                    f"{stage.stage_id}.{col.target_column}",
                    col.expression
                )

    return graph



#renderer new
from graphviz import Digraph


def render_pipeline_view(graph, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")

    # Dataset nodes
    for dataset in graph.datasets:
        dot.node(dataset, shape="box", style="filled", fillcolor="#E3F2FD")

    # Stage nodes
    for stage in graph.stages:
        dot.node(stage, shape="ellipse", style="filled", fillcolor="#FFF3E0")

    # Edges
    for src, tgt in graph.stage_edges:
        dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)



def render_stage_column_view(graph, stage_id, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")

    dot.node(stage_id, shape="ellipse", style="filled", fillcolor="#FFE0B2")

    for src, tgt, expr in graph.column_edges:
        if tgt.startswith(stage_id + "."):
            dot.node(src, shape="box", fillcolor="#E8F5E9", style="filled")
            dot.node(tgt, shape="box", fillcolor="#FFF9C4", style="filled")

            label = expr[:40] + "..." if len(expr) > 40 else expr
            dot.edge(src, tgt, label=label)

    dot.render(output_path, cleanup=True)

In [ ]:
#graph_model


class Graph:
    def __init__(self):
        self.source_tables = set()
        self.lookup_tables = set()
        self.processing_nodes = set()
        self.output_nodes = set()

        self.edges = []  # (source, target, label)

    def add_source(self, name):
        self.source_tables.add(name)

    def add_lookup(self, name):
        self.lookup_tables.add(name)

    def add_processing(self, name):
        self.processing_nodes.add(name)

    def add_output(self, name):
        self.output_nodes.add(name)

    def add_edge(self, src, tgt, label=None):
        self.edges.append((src, tgt, label))




#transform_summarizer

def summarize_expression(expression: str) -> str:
    if not expression:
        return ""

    expr = expression.lower()

    if "case" in expr:
        return "CASE Logic"
    if "if" in expr and "-99" in expr:
        return "Fallback if -99"
    if "if" in expr and "-97" in expr:
        return "Exception Handling"
    if "*" in expr:
        return "Derived (Multiplication)"
    if "+" in expr:
        return "Derived (Addition)"
    if "nvl" in expr:
        return "Null Handling"
    if "to_date" in expr:
        return "Date Conversion"

    return "Derived"



#graph_builder


from core.graph_model import Graph
from core.transform_summarizer import summarize_expression


def build_graph(lineage_response):
    graph = Graph()

    for stage in lineage_response.stages:

        graph.add_processing(stage.stage_id)

        # INPUTS
        for inp in stage.inputs:
            graph.add_source(inp["dataset_name"])
            graph.add_edge(inp["dataset_name"], stage.stage_id)

        # OUTPUTS
        for out in stage.outputs:
            graph.add_output(out["dataset_name"])
            graph.add_edge(stage.stage_id, out["dataset_name"])

        # COLUMN TRANSFORMATIONS (Skip DIRECT)
        for col in stage.column_lineage:
            if col.transformation_type != "DIRECT":
                label = summarize_expression(col.expression)

                for src in col.source_columns:
                    graph.add_edge(
                        src.split(".")[0],  # table only
                        stage.stage_id,
                        label
                    )

    return graph





#renderer


from graphviz import Digraph


def render_semantic_pipeline(graph, output_path):
    dot = Digraph(format="pdf")
    dot.attr(rankdir="LR")
    dot.attr(nodesep="1.0")
    dot.attr(ranksep="1.2")

    # ---------- SOURCE CLUSTER ----------
    with dot.subgraph(name="cluster_sources") as c:
        c.attr(label="Source Systems", style="rounded")
        for src in graph.source_tables:
            c.node(src, shape="box", style="filled", fillcolor="#1f2c3a")

    # ---------- PROCESSING CLUSTER ----------
    with dot.subgraph(name="cluster_processing") as c:
        c.attr(label="Processing", style="rounded")
        for proc in graph.processing_nodes:
            c.node(proc, shape="box", style="filled", fillcolor="#2d3e50")

    # ---------- OUTPUT CLUSTER ----------
    with dot.subgraph(name="cluster_outputs") as c:
        c.attr(label="Outputs", style="rounded")
        for out in graph.output_nodes:
            c.node(out, shape="box", style="filled", fillcolor="#34495e")

    # ---------- EDGES ----------
    for src, tgt, label in graph.edges:
        if label:
            dot.edge(src, tgt, label=label)
        else:
            dot.edge(src, tgt)

    dot.render(output_path, cleanup=True)



#system prompt


You are a deterministic ETL semantic lineage extractor.

Your goal is to produce a CLEAN transformation model suitable for visualization.

Rules:
1. Do NOT return raw SQL.
2. Do NOT return full expressions.
3. Summarize transformations in short semantic labels (max 6 words).
4. Skip direct column copies.
5. Classify transformations into:
   LOOKUP, DERIVED, CONDITIONAL, FILTER, AGGREGATION.
6. Identify lookup tables separately.
7. Identify exception outputs separately.
8. Return ONLY valid JSON.
9. Do NOT invent tables or joins.
10. If uncertain, omit the transformation.


#user prompt

Analyze the following pseudocode.

Extract a semantic transformation model for diagram generation.

Return JSON in this format:

{
  "stages": [
    {
      "stage_id": "",
      "stage_type": "",
      "source_tables": [],
      "lookup_tables": [],
      "output_tables": [],
      "transformations": [
        {
          "target_column": "",
          "source_tables": [],
          "transformation_category": "",
          "summary_label": ""
        }
      ],
      "output_conditions": [
        {
          "condition": "",
          "label": ""
        }
      ]
    }
  ]
}

Guidelines:
- Skip direct mappings.
- Summarize long logic.
- Keep labels concise.
- Do not include raw SQL.


#error

   main()
    ~~~~^^
  File "/home/admin/shraddha_code_repo/mermaid_llm/main.py", line 19, in main
    lineage = agent.extract_lineage(pseudocode)
  File "/home/admin/shraddha_code_repo/mermaid_llm/agent/llm_agent.py", line 38, in extract_lineage
    return LineageResponse(**parsed_json)
  File "/home/admin/shraddha_code_repo/mermaid_llm/mer_venv/lib/python3.13/site-packages/pydantic/main.py", line 250, in __init__
    validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)
pydantic_core._pydantic_core.ValidationError: 30 validation errors for LineageResponse
stages.0.inputs
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.outputs
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.column_lineage
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.joins
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.0.filters
  Field required [type=missing, input_value={'stage_id': 'ORA_Ext_Veh...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.inputs
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.outputs
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.column_lineage
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.joins
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.1.filters
  Field required [type=missing, input_value={'stage_id': 'Ora_StgVehi...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.inputs
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.outputs
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.column_lineage
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.joins
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.2.filters
  Field required [type=missing, input_value={'stage_id': 'HF_FACT_VOR...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.inputs
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.outputs
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.column_lineage
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.joins
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.3.filters
  Field required [type=missing, input_value={'stage_id': 'Tfm_LoadRec... for invalid SUPP_SK'}]}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.inputs
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.outputs
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.column_lineage
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.joins
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.4.filters
  Field required [type=missing, input_value={'stage_id': 'HF_SMR_VEHI...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.inputs
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.outputs
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.column_lineage
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.joins
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
stages.5.filters
  Field required [type=missing, input_value={'stage_id': 'Supp_VOR_Ex...'output_conditions': []}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing